In [0]:
%run ./config

In [0]:
from pyspark.sql import functions as F

def process_bronze_tabela(nome_tabela, arquivo):
    caminho = f"{VOLUME}/{arquivo}"
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("escape", '"')
          .csv(caminho))
    df = (df.withColumn("_arquivo_origem", F.lit(arquivo))
            .withColumn("_data_ingestao", F.current_timestamp()))
    destino = f"workspace.bronze.{nome_tabela}"
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(destino))
    return spark.table(destino).count()

In [0]:
def process_bronze(run_id=None):
    run_id = run_id or str(uuid.uuid4())
    resultados, auditoria = [], []
    for nome, arquivo in TABELAS.items():
        esperado = CONTAGENS_ESPERADAS[nome]
        inicio = datetime.now()
        try:
            linhas = process_bronze_tabela(nome, arquivo)
            status = "OK" if linhas == esperado else "DIVERGENTE"
            erro = None
        except Exception as e:
            linhas, status, erro = None, "ERRO", str(e)[:200]
        fim = datetime.now()
        resultados.append((nome, esperado, linhas, status, erro))
        auditoria.append((run_id, "bronze", nome, inicio, fim,
                          (fim - inicio).total_seconds(),
                          status, esperado, linhas, erro))
    record_audit(auditoria)
    return resultados